# Sesi 3: Single Object Detection — Training (v2, dengan Augmentasi + Partial Freeze)

**Bootcamp**: rubythalib.ai — mentor Daniel Syahputra

Notebook ini adalah versi konsolidasi dari `train.py` (dokumen sesi) + `training-object-detection.ipynb` (eksplorasi), sudah:
- Diverifikasi jalan tanpa error secara logika (dry-run dengan data sintetis, CPU)
- Menggunakan `mobilenet_v2(weights="DEFAULT")` (API terbaru, bukan `pretrained=True` yang sudah deprecated)
- `MixedLoss` dengan bobot yang bisa diatur (`weight_objectness`, `weight_localization`)

**Perubahan di v2** (hasil audit terhadap run pertama — IoU test 0.7186, tapi bounding box longgar pada kondisi langit gelap/dramatis):
1. **Data augmentation** (`ColorJitter`: brightness/contrast/saturation/hue) — **hanya diterapkan ke train set**, supaya model belajar bahwa objek yang sama tetap objek yang sama walau pencahayaan berbeda. Test set tetap pakai transform polos (resize + tensor) supaya evaluasi tidak bias.
2. **Partial backbone freeze** — 14 dari 19 blok awal MobileNetV2 dibekukan (`requires_grad=False`), cuma 5 blok terakhir + head yang di-fine-tune. Tujuannya menjaga fitur umum ImageNet (yang sudah tahan variasi pencahayaan) supaya tidak "tertimpa" oleh fine-tuning penuh pada dataset kecil (241 gambar).

Kedua perubahan ini eksperimental — belum tentu langsung menaikkan IoU test, tapi ditujukan untuk membuat bounding box lebih presisi di kondisi pencahayaan ekstrem (lihat temuan 7 sampel uji coba serving sebelumnya).

⚠️ **PENTING**: Jalankan notebook ini dengan **GPU runtime** (Runtime → Change runtime type → GPU) dan pastikan dataset sudah di-download/mount dari Google Drive sebelum menjalankan cell training penuh di paling bawah.


In [1]:
# Mount Google Drive & download dataset
from google.colab import drive
drive.mount('/content/drive')

# GANTI path ini sesuai lokasi dataset kamu setelah mount, atau download manual dari:
# https://drive.google.com/drive/folders/19ti0TK39f7SYJ06YQuxnXoLvVrp7xNhK?usp=sharing
DATASET_FOLDER = '/content/drive/MyDrive/single_object_detection'  # <- sesuaikan path ini


Mounted at /content/drive


In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import mobilenet_v2
from PIL import Image
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


Device: cpu


## 1. Dataset

`file_list` (opsional) dipakai supaya train set & test set bisa punya **transform berbeda** (augmentasi cuma di train) walau berasal dari folder yang sama — lihat pembagian train/test di bagian 7.

In [4]:
class ObjectDetectionDataset(Dataset):
    def __init__(self, image_folder, file_list=None, transform=None):
        self.image_folder = image_folder
        if file_list is not None:
            self.image_files = file_list
        else:
            self.image_files = [f for f in os.listdir(image_folder) if f.endswith(('png', 'jpg', 'jpeg'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_folder, img_name)
        annotation_path = os.path.join(self.image_folder, img_name.replace('jpg', 'txt'))

        image = Image.open(img_path).convert("RGB")
        img_width, img_height = image.size

        # default: tidak ada objek
        is_object = torch.tensor([0], dtype=torch.float32)
        bbox = torch.tensor([0, 0, 0, 0], dtype=torch.float32)

        if os.path.exists(annotation_path):
            with open(annotation_path, 'r') as f:
                lines = f.readlines()
                if len(lines) > 0:  # ada objek
                    x, y, w, h = map(float, lines[0].strip().split())
                    x /= img_width; y /= img_height; w /= img_width; h /= img_height
                    is_object = torch.tensor([1], dtype=torch.float32)
                    bbox = torch.tensor([x, y, w, h], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)
        return image, is_object, bbox


## 2. Model: MobileNetV2 backbone (partial freeze) + head kustom

`freeze_until=14` membekukan blok 0-13 (dari 19 blok `.features` MobileNetV2), cuma blok 14-18 + head yang ikut di-fine-tune. Tujuannya: fitur umum ImageNet di layer awal (deteksi tepi/tekstur dasar, biasanya sudah tahan variasi pencahayaan) tidak ikut "tertimpa" oleh fine-tuning penuh di atas dataset kecil (241 gambar).

In [5]:
class ObjectDetectionModel(nn.Module):
    def __init__(self, freeze_until: int = 14):
        super(ObjectDetectionModel, self).__init__()
        self.backbone = mobilenet_v2(weights="DEFAULT").features

        for i, block in enumerate(self.backbone):
            if i < freeze_until:
                for param in block.parameters():
                    param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1280, 512),   # 1280 = jumlah channel output MobileNetV2
            nn.ReLU(),
            nn.Linear(512, 5)       # 1 is_object + 4 bbox (x, y, w, h)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        is_object = torch.sigmoid(x[:, :1])
        bbox = torch.sigmoid(x[:, 1:])
        return is_object, bbox


In [6]:
# Cek jumlah parameter yang benar-benar dilatih vs dibekukan
_tmp_model = ObjectDetectionModel()
_trainable = sum(p.numel() for p in _tmp_model.parameters() if p.requires_grad)
_frozen = sum(p.numel() for p in _tmp_model.parameters() if not p.requires_grad)
print(f"Trainable params: {_trainable:,}")
print(f"Frozen params:    {_frozen:,}")
del _tmp_model


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 132MB/s]

Trainable params: 2,339,781
Frozen params:    542,528


## 3. IoU (Intersection over Union) — metrik evaluasi

In [7]:
def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    x_left = max(x1 - w1/2, x2 - w2/2)
    y_top = max(y1 - h1/2, y2 - h2/2)
    x_right = min(x1 + w1/2, x2 + w2/2)
    y_bottom = min(y1 + h1/2, y2 + h2/2)
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    intersection = (x_right - x_left) * (y_bottom - y_top)
    union = w1*h1 + w2*h2 - intersection
    return intersection / union


## 4. Loss gabungan: BCE (objectness) + MSE (localization)

In [8]:
class MixedLoss(nn.Module):
    def __init__(self, weight_objectness: float = 1.0, weight_localization: float = 1.0):
        super(MixedLoss, self).__init__()
        self.bce_loss = nn.BCELoss()
        self.mse_loss = nn.MSELoss()
        self.weight_o = weight_objectness
        self.weight_l = weight_localization

    def forward(self, pred_object, true_object, pred_bbox, true_bbox):
        loss_object = self.bce_loss(pred_object, true_object)
        loss_bbox = self.mse_loss(pred_bbox, true_bbox)
        return self.weight_o * loss_object + self.weight_l * loss_bbox


## 5. Sanity check (jalankan dulu sebelum training penuh)

Cell-cell ini memastikan tiap komponen bekerja sebelum training 30 epoch penuh: cek 1 sample dataset, cek shape output model dengan dummy batch, cek loss dengan dummy ground truth.

In [9]:
dataset_check = ObjectDetectionDataset(image_folder=DATASET_FOLDER)
image, is_object, bbox = next(iter(dataset_check))
print("Sample check -> is_object:", is_object, "| bbox:", bbox)


Sample check -> is_object: tensor([1.]) | bbox: tensor([0.5106, 0.5153, 0.6627, 0.6608])


In [10]:
model_check = ObjectDetectionModel().to(device)
input_tensor = torch.rand(16, 3, 224, 224).to(device)
objectness, bboxes = model_check(input_tensor)
print("objectness shape:", objectness.shape)   # harus torch.Size([16, 1])
print("bboxes shape:", bboxes.shape)           # harus torch.Size([16, 4])


objectness shape: torch.Size([16, 1])
bboxes shape: torch.Size([16, 4])


In [11]:
dummy_gt_object = torch.ones(16, 1).to(device)
dummy_gt_local = torch.rand(16, 4).to(device)
test_loss_fn = MixedLoss(weight_objectness=0.5, weight_localization=1.0)
loss_check = test_loss_fn(objectness, dummy_gt_object, bboxes, dummy_gt_local)
print("Sanity-check loss value:", loss_check.item())
del model_check, input_tensor, objectness, bboxes, dummy_gt_object, dummy_gt_local, loss_check


Sanity-check loss value: 0.4132084250450134


## 6. Training & validation loop (dengan tracking IoU)

In [12]:
def train_model(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=10):
    model.train()
    best_iou = 0.0
    history = {"loss": [], "iou_train": [], "iou_test": []}
    for epoch in range(num_epochs):
        running_loss, total_iou_train, total_iou_test = 0.0, 0.0, 0.0

        for images, is_object, bboxes in train_loader:
            images, is_object, bboxes = images.to(device), is_object.to(device), bboxes.to(device)
            optimizer.zero_grad()
            pred_object, pred_bbox = model(images)
            loss = criterion(pred_object, is_object, pred_bbox, bboxes)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            for i in range(len(bboxes)):
                if is_object[i] > 0.5:
                    total_iou_train += calculate_iou(pred_bbox[i].detach().cpu().numpy(), bboxes[i].cpu().numpy())

        avg_iou_train = total_iou_train / len(train_loader.dataset)
        avg_loss = running_loss/len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, IoU (Train): {avg_iou_train:.4f}")

        model.eval()
        with torch.no_grad():
            for images, is_object, bboxes in test_loader:
                images, is_object, bboxes = images.to(device), is_object.to(device), bboxes.to(device)
                pred_object, pred_bbox = model(images)
                for i in range(len(bboxes)):
                    if is_object[i] > 0.5:
                        total_iou_test += calculate_iou(pred_bbox[i].cpu().numpy(), bboxes[i].cpu().numpy())

        avg_iou_test = total_iou_test / len(test_loader.dataset)
        print(f"IoU (Test): {avg_iou_test:.4f}")

        history["loss"].append(avg_loss)
        history["iou_train"].append(avg_iou_train)
        history["iou_test"].append(avg_iou_test)

        torch.save(model.state_dict(), 'last.pt')
        if avg_iou_test > best_iou:
            best_iou = avg_iou_test
            torch.save(model.state_dict(), 'best.pt')
            print(f"  -> best.pt disimpan (IoU test: {best_iou:.4f})")
        model.train()

    return history, best_iou


## 7. Split Train/Test + Augmentasi (hanya di train) + Jalankan Training Penuh

Beda dari v1: file di-split **dulu** jadi daftar nama file train/test, baru masing-masing dibungkus `ObjectDetectionDataset` dengan transform berbeda. Ini supaya `ColorJitter` (augmentasi warna/pencahayaan) **tidak ikut diterapkan ke test set** — kalau test set ikut teraugmentasi, angka IoU test jadi tidak representatif untuk evaluasi asli.

Sesuaikan `batch_size` dan `num_epochs` di bawah sesuai kapasitas GPU Colab-mu. Nilai default mengikuti versi notebook eksplorasi (`batch_size=128`) — turunkan ke 16 (versi dokumen awal) kalau kena out-of-memory.

In [13]:
import random

batch_size = 128
num_epochs = 30
learning_rate = 0.001

# --- Split nama file dulu (bukan random_split objek Dataset) ---
all_files = [f for f in os.listdir(DATASET_FOLDER) if f.endswith(('png', 'jpg', 'jpeg'))]
random.seed(42)
random.shuffle(all_files)

split_idx = int(0.8 * len(all_files))
train_files, test_files = all_files[:split_idx], all_files[split_idx:]
print(f"Total data: {len(all_files)} | Train: {len(train_files)} | Test: {len(test_files)}")

# --- Transform terpisah: augmentasi CUMA di train, test tetap polos ---
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
# Catatan: RandomHorizontalFlip sengaja TIDAK dipakai -- flip akan mengubah posisi x
# bounding box juga (x_baru = lebar - x_lama), yang berarti perlu logika tambahan untuk
# ikut membalik label bbox. Di luar scope perbaikan ini (fokusnya sensitivitas warna/cahaya).

train_dataset = ObjectDetectionDataset(DATASET_FOLDER, file_list=train_files, transform=train_transform)
test_dataset = ObjectDetectionDataset(DATASET_FOLDER, file_list=test_files, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

model = ObjectDetectionModel(freeze_until=14).to(device)
criterion = MixedLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)

history, best_iou = train_model(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=num_epochs)
print(f"\nSelesai. Best IoU (test): {best_iou:.4f}")


Total data: 241 | Train: 192 | Test: 49
Epoch [1/30], Loss: 0.5637, IoU (Train): 0.4743
IoU (Test): 0.5644
  -> best.pt disimpan (IoU test: 0.5644)
Epoch [2/30], Loss: 0.2233, IoU (Train): 0.5544
IoU (Test): 0.6974
  -> best.pt disimpan (IoU test: 0.6974)
Epoch [3/30], Loss: 0.1177, IoU (Train): 0.6796
IoU (Test): 0.6224
Epoch [4/30], Loss: 0.0957, IoU (Train): 0.6337
IoU (Test): 0.6065
Epoch [5/30], Loss: 0.0849, IoU (Train): 0.6631
IoU (Test): 0.6107
Epoch [6/30], Loss: 0.0763, IoU (Train): 0.6592
IoU (Test): 0.6378
Epoch [7/30], Loss: 0.0738, IoU (Train): 0.6743
IoU (Test): 0.6576
Epoch [8/30], Loss: 0.0674, IoU (Train): 0.6791
IoU (Test): 0.6843
Epoch [9/30], Loss: 0.0710, IoU (Train): 0.7242
IoU (Test): 0.6712
Epoch [10/30], Loss: 0.0562, IoU (Train): 0.7154
IoU (Test): 0.6859
Epoch [11/30], Loss: 0.0589, IoU (Train): 0.7558
IoU (Test): 0.6878
Epoch [12/30], Loss: 0.0597, IoU (Train): 0.7142
IoU (Test): 0.6912
Epoch [13/30], Loss: 0.0589, IoU (Train): 0.7405
IoU (Test): 0.6681
Epo

## 8. Simpan checkpoint ke Google Drive (WAJIB — jangan skip)

`best.pt` dan `last.pt` di atas kesimpan ke `/content/` — folder sementara Colab yang **hilang begitu runtime disconnect**. Cell ini menyalin keduanya ke Google Drive supaya tidak hilang.

In [14]:
import shutil

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/single_object_detection/checkpoints'  # <- sesuaikan kalau perlu
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

shutil.copy('best.pt', os.path.join(DRIVE_CHECKPOINT_DIR, 'best.pt'))
shutil.copy('last.pt', os.path.join(DRIVE_CHECKPOINT_DIR, 'last.pt'))

print(f"Checkpoint disalin ke Google Drive: {DRIVE_CHECKPOINT_DIR}")
print(f"Best IoU (test) yang tersimpan: {best_iou:.4f}")


Checkpoint disalin ke Google Drive: /content/drive/MyDrive/single_object_detection/checkpoints
Best IoU (test) yang tersimpan: 0.7276


## 9. Catatan hasil (isi manual setelah training selesai)

Setelah training selesai, salin angka `best_iou` dan grafik `history` di atas untuk dipakai di README GitHub dan blog Hashnode — **jangan pakai angka estimasi**, harus dari hasil run ini.